# Import Libraries

In [ ]:
!pip install -qU \
langchain \
langchain-community \
langchain-text-splitters \
faiss-cpu \
pypdf

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
print(os.listdir("/content"))

# Load pdf file

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH = "/content/24EG107A46 - P CHARAN KUMAR REDDY (1).pdf"
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

In [ ]:
print(f"Total Pages: {len(documents)}")

In [ ]:
documents[0]

In [ ]:
print(documents[0].page_content)

In [ ]:
import uuid
import re
import os

# cleaning the data

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean text extracted from PDF.
    """
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [ ]:
processed_documents = []
for doc in documents:
    cleaned_text = clean_text(doc.page_content)
    metadata = {
        "document_id": str(uuid.uuid4()),
        "source": os.path.basename(doc.metadata["source"]),
        "page_number": doc.metadata["page"],
        "character_count": len(cleaned_text),
        "word_count": len(cleaned_text.split())
    }
    doc.page_content = cleaned_text
    doc.metadata = metadata
    processed_documents.append(doc)

In [ ]:
print(f"Processed Documents : {len(processed_documents)}")

In [ ]:
processed_documents[0].metadata

In [ ]:
for doc in processed_documents:
    print("=" * 70)
    print("Document ID    :", doc.metadata["document_id"])
    print("Source         :", doc.metadata["source"])
    print("Page Number    :", doc.metadata["page_number"])
    print("Characters     :", doc.metadata["character_count"])
    print("Words          :", doc.metadata["word_count"])

# chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=30,separators=["\n\n","\n",". "," ",""])

In [ ]:
chunks = text_splitter.split_documents(processed_documents)
print("Total Chunks:", len(chunks))

In [ ]:
for index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = str(uuid.uuid4())
    chunk.metadata["chunk_index"] = index

In [ ]:
chunks[0]

In [ ]:
for chunk in chunks:
    print("="*80)
    print("Chunk Index :", chunk.metadata["chunk_index"])
    print()
    print(chunk.page_content)

In [ ]:
for chunk in chunks:
    print("="*80)
    print("Chunk Index :", chunk.metadata["chunk_index"])
    print()
    print(chunk.page_content)

# Embedding

In [ ]:
!pip install -q -U langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    }
)

In [ ]:
vector = embedding_model.embed_query("What is Artificial Intelligence?")
print(type(vector))
print(len(vector))

In [ ]:
sample_vector = embedding_model.embed_query(chunks[0].page_content)
print(len(sample_vector))

In [ ]:
chunk_vectors = embedding_model.embed_documents([chunk.page_content for chunk in chunks])
print("Total Embeddings :", len(chunk_vectors))
print("Embedding Dimension :", len(chunk_vectors[0]))

In [ ]:
print(chunk_vectors[0][:10])

# faiss vector database

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [ ]:
print(type(vectorstore))

In [ ]:
print("Total Indexed Chunks :", vectorstore.index.ntotal)

In [ ]:
print(vectorstore.index)

In [ ]:
print(vectorstore.docstore._dict)

In [ ]:
vectorstore.save_local("faiss_index")

In [ ]:
loaded_vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
print(loaded_vectorstore.index.ntotal)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3
    }
)

In [ ]:
print(type(retriever))

# query

In [ ]:
query = "What are the skills of Charan?"
results = retriever.invoke(query)
print("Retrieved Chunks :", len(results))

In [ ]:
import re

In [ ]:
def clean_query(query: str) -> str:
    """
    Clean and normalize user query.
    """
    query = query.strip()
    query = re.sub(r"\s+", " ", query)
    return query

# Query processing

In [ ]:
user_query = "   What     are     Charan's     Skills?   "
processed_query = clean_query(user_query)
print(processed_query)

In [ ]:
query_vector = embedding_model.embed_query(
    processed_query
)
print(len(query_vector))

In [ ]:
retrieved_docs = retriever.invoke(
    processed_query
)

print(len(retrieved_docs))

In [ ]:
for i, doc in enumerate(retrieved_docs):

    print("=" * 80)

    print(f"Document {i+1}")

    print()

    print(doc.page_content)

    print()

    print(doc.metadata)

# Retriveing model

In [ ]:
!pip install rank_bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever

In [ ]:
bm25_retriever = BM25Retriever.from_documents(chunks)

In [ ]:
bm25_retriever.k = 3

In [ ]:
query = "what is the name of person?"

bm25_results = bm25_retriever.invoke(query)

print(len(bm25_results))

In [ ]:
for i, doc in enumerate(bm25_results):

    print("="*80)

    print(f"BM25 Result {i+1}")

    print(doc.page_content)

In [ ]:
faiss_results = retriever.invoke(query)

print(len(faiss_results))

In [ ]:
for i, doc in enumerate(faiss_results):

    print("="*80)

    print(f"FAISS Result {i+1}")

    print(doc.page_content)

In [ ]:
hybrid_results = []

seen = set()

for doc in bm25_results + faiss_results:

    text = doc.page_content

    if text not in seen:

        hybrid_results.append(doc)

        seen.add(text)

print("Hybrid Results:", len(hybrid_results))

In [ ]:
for i, doc in enumerate(hybrid_results):

    print("="*80)

    print(f"Hybrid Result {i+1}")

    print()

    print(doc.page_content)

# Reranker

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

In [ ]:
pairs = [
    (query, doc.page_content)
    for doc in hybrid_results
]

In [ ]:
scores = reranker.predict(pairs)

In [ ]:
reranked_results = list(
    zip(hybrid_results, scores)
)

In [ ]:
reranked_results = sorted(
    reranked_results,
    key=lambda x: x[1],
    reverse=True
)

In [ ]:
for rank, (doc, score) in enumerate(reranked_results, start=1):

    print("=" * 80)

    print(f"Rank : {rank}")

    print(f"Score : {score:.4f}")

    print()

    print(doc.page_content[:400])

In [ ]:
TOP_K = 3

top_documents = [
    doc
    for doc, score in reranked_results[:TOP_K]
]

In [ ]:
print(len(top_documents))

In [ ]:
unique_documents = []

seen = set()

for doc in top_documents:

    text = doc.page_content.strip()

    if text not in seen:

        unique_documents.append(doc)

        seen.add(text)

print("Unique Documents :", len(unique_documents))

In [ ]:
context = ""

for i, doc in enumerate(unique_documents, start=1):

    context += f"""
==============================
Document {i}

Source : {doc.metadata['source']}

Page : {doc.metadata['page_number']}

Content:
{doc.page_content}

"""

In [ ]:
print(context)

In [ ]:
question = "What programming languages does Charan know?"

# Query expansion

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
You are an expert AI assistant.

Answer ONLY using the supplied context.

If the answer is not present in the context,
reply exactly:

"I couldn't find that information in the documents."

Context:
{context}

Question:
{question}
"""
)

In [ ]:
print(type(prompt))

In [ ]:
!pip install -q -U langchain-groq

# Llm

In [ ]:
from langchain_groq import ChatGroq
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
chain = prompt | llm

In [ ]:
chain = prompt_template | llm

response = chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

In [ ]:
final_response = {
    "question": question,
    "answer": response.content,
    "sources": unique_sources
}

In [ ]:
from pprint import pprint

pprint(final_response)